# Exploratory Data Analysis & Spam Classifier Project
This notebook covers the complete step-by-step pipeline of exploratory data analysis, preprocessing, feature engineering, modeling, and comparison of multiple classification algorithms for classifying SMS/Email messages into **Spam** or **Ham**.

## 1. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
import nltk
from wordcloud import WordCloud
from collections import Counter

# Set plots design
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12
})

## 2. Load and Clean Dataset

In [ ]:
url = "https://raw.githubusercontent.com/mohitgupta-omg/Kaggle-SMS-Spam-Collection-Dataset-/master/spam.csv"
df = pd.read_csv(url, encoding="latin-1")

# Remove extra columns
df.drop(columns=[col for col in df.columns if col.startswith("Unnamed")], inplace=True)

# Rename columns
df.columns = ["label", "message"]

# Clean duplicates
print(f"Initial shape: {df.shape}")
df.drop_duplicates(keep="first", inplace=True)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"Cleaned shape: {df.shape}")

# Binary label
df["label_num"] = df["label"].map({"ham": 0, "spam": 1})
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Message character length, word counts and sentence counts
df["length"] = df["message"].apply(len)
df["word_count"] = df["message"].apply(lambda x: len(x.split()))
df["sent_count"] = df["message"].apply(lambda x: len(re.split(r'[.!?]+', x)) - 1)
df.groupby("label")[["length", "word_count", "sent_count"]].mean()

In [ ]:
# Class distribution
plt.figure(figsize=(6, 4))
df["label"].value_value = df["label"].value_counts()
sns.barplot(x=df["label"].value_counts().index, y=df["label"].value_counts().values, palette="viridis")
plt.title("Class Distribution (Ham vs Spam)")
plt.ylabel("Count")
plt.show()

In [ ]:
# Length distribution histogram
plt.figure(figsize=(12, 5))
sns.histplot(df[df["label_num"] == 0]["length"], bins=50, color="green", label="Ham", kde=True, alpha=0.5)
sns.histplot(df[df["label_num"] == 1]["length"], bins=50, color="red", label="Spam", kde=True, alpha=0.5)
plt.xlim(0, 300)
plt.title("Message Length (Characters) Comparison")
plt.xlabel("Length")
plt.ylabel("Frequency")
plt.legend()
plt.show()

In [ ]:
# Correlation analysis
plt.figure(figsize=(6, 4))
sns.heatmap(df[["label_num", "length", "word_count", "sent_count"]].corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Map of Engineered Text Features")
plt.show()

## 4. Text Preprocessing

In [ ]:
import string
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    # Lowercase
    text = text.lower()
    # Tokenize
    try:
        words = nltk.word_tokenize(text)
    except:
        words = text.split()
    
    stop_words = set(stopwords.words('english'))
    punctuation = set(string.punctuation)
    stemmer = PorterStemmer()
    
    cleaned = []
    for w in words:
        cw = "".join(char for char in w if char not in punctuation)
        if cw and cw not in stop_words:
            cleaned.append(stemmer.stem(cw))
    return " ".join(cleaned)

df["processed_message"] = df["message"].apply(preprocess_text)
df[["message", "processed_message"]].head()

## 5. Word Clouds and Common Words

In [ ]:
spam_words = " ".join(df[df["label_num"] == 1]["processed_message"])
ham_words = " ".join(df[df["label_num"] == 0]["processed_message"])

wc_spam = WordCloud(width=800, height=400, background_color="white", colormap="Reds").generate(spam_words)
wc_ham = WordCloud(width=800, height=400, background_color="white", colormap="Greens").generate(ham_words)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(wc_ham, interpolation="bilinear")
axes[0].set_title("Ham Word Cloud")
axes[0].axis("off")

axes[1].imshow(wc_spam, interpolation="bilinear")
axes[1].set_title("Spam Word Cloud")
axes[1].axis("off")

plt.show()

## 6. Model Training, Comparison & Evaluation

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

X = df["processed_message"]
y = df["label_num"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

tfidf = TfidfVectorizer(max_features=5000, min_df=2)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

models = {
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(random_state=42, dual=False),
    "Logistic Regression": LogisticRegression(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred)
    })

df_res = pd.DataFrame(results)
df_res

## 7. Plotting Model Metrics Comparison

In [ ]:
df_melted = pd.melt(df_res, id_vars="Model", var_name="Metric", value_name="Score")
plt.figure(figsize=(12, 6))
sns.barplot(x="Model", y="Score", hue="Metric", data=df_melted, palette="muted")
plt.ylim(0.8, 1.02)
plt.title("Evaluation Metrics Comparison")
plt.show()